# Step 3 — reproducible five-fold training

Attach the step-2c v2 cache and the label output. Install `iterative-stratification` in the Kaggle training environment. Select one fold first to measure GPU memory and runtime at 336px. Checkpoint selection uses pseudo-validation soft-label BCE; gold is evaluated only after selection. OOF AUC measures agreement with noisy labels and is affected by epoch selection; it is not a human-label generalization estimate. EfficientNet is the baseline; select DINOv3-small via BACKBONE for a controlled stronger-pretraining comparison.

## 0 · Config

In [ ]:
# Training requires Internet for pretrained weights and this small split helper.
import importlib.util, subprocess, sys
if importlib.util.find_spec('iterstrat') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','iterative-stratification==0.1.9'])

In [ ]:
import os
os.environ["KNEE_FOLDS"] = "[0]"


In [ ]:
# Run cells in order; no working-directory changes are needed.

In [ ]:
import os, gc, math, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ---- inputs ------------------------------------------------------------------
CACHE_DIR = os.environ.get("KNEE_CACHE_DIR", "/kaggle/input/notebooks/shreyanshgoyaliitb/knee-mri-step2c-cache")
LABELS = os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/shreyanshgoyaliitb/compi-label-generation/train_targets.csv")
COMP_DIR  = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT       = "/kaggle/working"
# -----------------------------------------------------------------------------

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

_meta_path = os.path.join(CACHE_DIR, 'cache_meta.json')
assert os.path.exists(_meta_path), 'Attach the rebuilt step-2c v2 output and set CACHE_DIR'
CACHE_META = json.load(open(_meta_path))
assert CACHE_META.get('version') == 'knee-2c-v2', 'Rebuild the step-2c cache with v2 preprocessing'
PLANES = CACHE_META['slots']
NORMALIZATION = 'pretrained'
DROPOUT = .4
N_FOLDS = 5
FOLDS_TO_RUN = json.loads(os.environ.get("KNEE_FOLDS", "[0,1,2,3,4]"))  # one fold first on Kaggle
GROUP_COL = None  # automatically populated below if patient IDs are supplied
RESUME = False  # copy both fold_N.pt and last_fold_N.pt to OUT before enabling
ENCODER_LR_SCALE = .3
BACKBONE = os.environ.get("KNEE_BACKBONE", "tf_efficientnet_b0.ns_jft_in1k")
# Stronger-pretraining experiment: vit_small_patch16_dinov3.lvd1689m; compare the same folds.
IMG         = 224
N_SLICES    = 16       # overridden from the cache below
T_TRAIN     = 10       # slices sampled per plane per step (augmentation + speed)
BATCH       = 4        # one T4 saturates here: batch 8 gave 364 img/s vs 369
EPOCHS      = 6
LR          = 3e-4
WD          = 1e-2
SEED        = 42
NUM_WORKERS = 4        # Kaggle gives 4 cores regardless of accelerator

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune

print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
if DEV == "cuda" and torch.cuda.device_count() > 1:
    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "
          f"(DataParallel benchmarked 13x slower here)")
ACCUM_STEPS = 1
if BACKBONE.startswith('vit_'):
    BATCH, ACCUM_STEPS = 1, 4  # effective batch 4; lower activation memory

## 1 · Data

In [ ]:
index = pd.read_csv(os.path.join(CACHE_DIR, "cache_index.csv"))
cache = np.load(os.path.join(CACHE_DIR, "cache.npy"), mmap_mode="r")
print("cache", cache.shape, cache.dtype, f"{cache.nbytes/1e9:.1f} GB")

# Geometry comes from the cache, not the constants -- rebuilding the cache at a
# different depth must never silently corrupt training.
_, N_PLANES, N_SLICES, H_C, W_C = cache.shape
assert H_C == W_C, "non-square cache"
if (H_C, N_PLANES) != (IMG, len(PLANES)):
    print(f"  overriding config: IMG {IMG}->{H_C}, planes {len(PLANES)}->{N_PLANES}")
IMG = H_C
T_TRAIN = min(T_TRAIN, N_SLICES)
print(f"  geometry: {N_PLANES} planes x {N_SLICES} slices x {IMG}px | T_TRAIN={T_TRAIN}")

# Labels: accept train_targets.csv (has 'source') or pseudo_labels.csv (doesn't).
targets = pd.read_csv(LABELS)
print(f"\nlabels: {os.path.basename(LABELS)}  {targets.shape}")

if "source" not in targets.columns:
    print("  no 'source' column -> pseudo-only file; recovering gold rows from train.csv")
    tr = pd.read_csv(os.path.join(COMP_DIR, "train.csv"))
    for c in LABEL_COLS:
        tr[c] = pd.to_numeric(tr.get(c), errors="coerce")
    gold = tr.loc[tr[LABEL_COLS].notna().all(axis=1), ["StudyInstanceUID", *LABEL_COLS]].copy()
    gold["source"] = "gold"
    ps = targets[["StudyInstanceUID", *LABEL_COLS]].copy()
    ps["source"] = "pseudo"
    ps = ps[~ps["StudyInstanceUID"].isin(set(gold["StudyInstanceUID"]))]
    targets = pd.concat([gold, ps], ignore_index=True)
    print(f"  gold {len(gold)} + pseudo {len(ps)} = {len(targets)}")

targets[LABEL_COLS] = targets[LABEL_COLS].astype("float32")
assert targets["StudyInstanceUID"].is_unique, "duplicate studies in the label file"

df = index[index["n_present"] > 0].merge(
        targets[["StudyInstanceUID", *LABEL_COLS, "source"]],
        on="StudyInstanceUID", how="inner")
print(f"\nusable studies: {len(df):,}  ({df['source'].value_counts().to_dict()})")

y_all = df[LABEL_COLS].to_numpy(np.float32)
print(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")
print("\npositive rate (observed cells only):")
print(pd.Series(np.nanmean(y_all, axis=0), index=LABEL_COLS).round(3).to_string())
assert cache.dtype == np.uint8 and len(index) == len(cache)
assert len(PLANES) == N_PLANES and IMG == CACHE_META['img'] and N_SLICES == CACHE_META['n_slices']
assert index['StudyInstanceUID'].is_unique and index['row'].is_unique
assert index['row'].between(0,len(cache)-1).all()
assert df['source'].isin(['gold','pseudo']).all(), 'Unknown label source'
assert ((np.isnan(y_all)) | ((y_all>=0)&(y_all<=1))).all(), 'Targets outside [0,1]'
# Recover human labels from authoritative train.csv even when a source column was supplied.
original = pd.read_csv(os.path.join(COMP_DIR,'train.csv'))
original[LABEL_COLS] = original[LABEL_COLS].apply(pd.to_numeric,errors='coerce')
gold_table = original.loc[original[LABEL_COLS].notna().any(axis=1)].set_index('StudyInstanceUID')
mask_gold = df['StudyInstanceUID'].isin(gold_table.index)
df.loc[mask_gold,'source'] = 'gold'
df.loc[mask_gold,LABEL_COLS] = gold_table.loc[df.loc[mask_gold,'StudyInstanceUID'],LABEL_COLS].to_numpy()
assert not (df['source'].eq('gold') & ~mask_gold).any(), 'Gold rows absent from train.csv'
# If patient IDs exist, keep both knees and repeated studies in one fold; exclude gold patients.
for candidate in ('PatientID','patient_id','PatientId'):
    if candidate in original and original[candidate].notna().all():
        df = df.merge(original[['StudyInstanceUID',candidate]],on='StudyInstanceUID',validate='one_to_one')
        GROUP_COL = candidate
        gold_groups = set(df.loc[df['source'].eq('gold'),candidate])
        df = df[~(df['source'].eq('pseudo') & df[candidate].isin(gold_groups))].reset_index(drop=True)
        break
if GROUP_COL is None: print('No patient identifier available: folds are study-disjoint only.')
df = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)

In [ ]:
"""Fold creation and metrics with explicit gold exclusion."""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


def assign_folds(df, label_cols, n_splits=5, seed=42, group_col=None):
    folds = np.full(len(df), -1, dtype=int)
    pseudo = np.flatnonzero(df['source'].eq('pseudo').to_numpy())
    assert len(pseudo) >= n_splits
    y = df.iloc[pseudo][label_cols].to_numpy(float)
    # Balance missingness as well as positive findings. No gold target enters the split.
    strata = np.concatenate([np.nan_to_num(y, nan=0) >= .5, np.isfinite(y)], axis=1).astype(int)
    groups = df.iloc[pseudo][group_col].astype(str).to_numpy() if group_col else df.iloc[pseudo]['StudyInstanceUID'].to_numpy()
    if len(np.unique(groups)) < len(groups):
        splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = splitter.split(pseudo, groups=groups)
    else:
        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = splitter.split(pseudo, strata)
    for fold, (_, val) in enumerate(splits): folds[pseudo[val]] = fold
    assert (folds[pseudo] >= 0).all()
    assert (folds[df['source'].eq('gold')] == -1).all()
    return folds


def macro_auc(y, scores, label_cols):
    aucs = {}
    for k, name in enumerate(label_cols):
        ok = np.isfinite(y[:, k]) & np.isfinite(scores[:, k])
        binary = (y[ok, k] >= .5).astype(int)
        aucs[name] = roc_auc_score(binary, scores[ok, k]) if len(np.unique(binary)) == 2 else np.nan
    valid = [v for v in aucs.values() if np.isfinite(v)]
    return (float(np.mean(valid)) if valid else float('nan')), aucs


def rank_blend(predictions):
    arrays = [np.asarray(p, dtype=float) for p in predictions]
    assert arrays and all(a.shape == arrays[0].shape for a in arrays)
    assert all(np.isfinite(a).all() for a in arrays)
    return np.mean([pd.DataFrame(a).rank(pct=True, method='average').to_numpy() for a in arrays], axis=0)

In [ ]:
import cv2

class KneeDS(Dataset):
    def __init__(self, df, rows, train):
        self.df = df.reset_index(drop=True)
        self.rows = rows
        self.train = train
        self.y = self.df[LABEL_COLS].to_numpy(np.float32)
        self.present = self.df[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)
        self.row_of = self.df["row"].to_numpy(int)
        self.cache = None                      # memmap opened lazily, per worker

    def __len__(self):
        return len(self.rows)

    def _aug(self, vol):
        # NO horizontal or vertical flip: a horizontal flip in the coronal plane swaps
        # medial and lateral, which would invert four of the twelve labels.
        h, w = vol.shape[-2:]
        ang = np.random.uniform(-12, 12)
        sc  = np.random.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((w/2, h/2), ang, sc)
        M[:, 2] += np.random.uniform(-0.06, 0.06, 2) * w
        out = np.empty_like(vol)
        for i in range(vol.shape[0]):
            out[i] = cv2.warpAffine(vol[i], M, (w, h), flags=cv2.INTER_LINEAR,
                                    borderValue=0)
        if np.random.rand() < 0.5:
            out = np.clip(out.astype(np.float32) * np.random.uniform(0.85, 1.15)
                          + np.random.uniform(-15, 15), 0, 255).astype(np.uint8)
        return out

    def __getitem__(self, i):
        if self.cache is None:
            self.cache = np.load(os.path.join(CACHE_DIR, "cache.npy"), mmap_mode="r")
        j = self.rows[i]
        r = self.row_of[j]
        present = self.present[j]

        if self.train:
            # Slice the memmap BEFORE materialising, so the discarded slices are
            # never read off disk at all (~37% less I/O per sample).
            avail = self.cache.shape[2]
            t = min(T_TRAIN, avail)
            t0 = np.random.randint(0, avail - t + 1)
            vol = np.asarray(self.cache[r][:, t0:t0 + t])
            vol = np.stack([self._aug(v) if present[s] else v
                            for s, v in enumerate(vol)])
        else:
            vol = np.asarray(self.cache[r])          # eval uses every slice

        return (torch.from_numpy(np.array(vol, copy=True, order='C')),
                torch.from_numpy(present.copy()),
                torch.from_numpy(self.y[j].copy()))


def collate(batch):
    xs, ps, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ps), torch.stack(ys)

## 2 · Model

`make_neighbours` gives each slice a 3-channel view of its neighbours, so an
ImageNet-pretrained 2D backbone sees through-plane context for free.

`LabelAttnPool` gives **each finding its own weighting over planes**. With a single
shared attention the axial plane collapsed to 0.001 — nine of twelve findings live in
sagittal and coronal, so the shared scorer starved the minority, and PF OA and
Synovitis (the axial-dependent ones) were the two weakest columns.

In [ ]:
"""Shared model architecture for training and offline inference."""
import torch
import torch.nn as nn
import timm

def make_neighbours(x):
    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    # Attention over slices within one plane.
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):               # x (B,T,D)
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    # Per-label attention over planes: each finding picks the view it needs.
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        a = self.score(x)                          # (B,S,L)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)                       # over planes, per label
        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)
        return z, a


class KneeNet(nn.Module):
    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,
                 pretrained=True, drop=0.3, img=224, normalization="legacy",
                 encoder_chunk=64):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.head_w, std=0.02)

    def forward(self, x, present):
        # autocast entered HERE, not in the training loop: it is thread-local, so a
        # loop-level autocast never reaches a threaded forward. Verified reaching the
        # encoder as float16.
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div(255.)   # ImageNet grey stats
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)

            x3 = (x3 - self.input_mean) / self.input_std
            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.
            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:
                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])
            else:
                features = self.enc(x3)
            f = features.reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)

            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            z, plane_att = self.plane_pool(v, m)
            z = self.drop(z)
            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, plane_att

## 3 · Loss and metric

In [ ]:
def masked_bce(logits, targets):
    # BCE over (B,12) targets that may contain NaN. NaN cells get exactly zero
    # gradient, so a study the extractor abstained on still trains the other heads.
    mask = ~torch.isnan(targets)
    if not mask.any():
        return logits.sum() * 0.0
    loss = F.binary_cross_entropy_with_logits(
        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")
    return (loss * mask).sum() / mask.sum()

## 4 · Train

In [ ]:
# Training, checkpoint selection, and fold exports are implemented below.

In [ ]:
"""Five-fold training runtime, embedded in training.ipynb."""
def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)
    cv2.setNumThreads(0)


def make_loader(rows, training, seed):
    return DataLoader(KneeDS(df, rows, training), batch_size=BATCH,
                      shuffle=training, num_workers=NUM_WORKERS,
                      collate_fn=collate, pin_memory=DEV=='cuda', drop_last=False,
                      persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(seed),
                      **({'prefetch_factor': 2} if NUM_WORKERS else {}))


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, present, y in loader:
        logits, _ = model(x.to(DEV, non_blocking=True), present.to(DEV, non_blocking=True))
        ps.append(logits.float().sigmoid().cpu().numpy()); ys.append(y.numpy())
    if not ys: return np.empty((0,len(LABEL_COLS))), np.empty((0,len(LABEL_COLS)))
    return np.concatenate(ys), np.concatenate(ps)


def atomic_torch_save(obj, path):
    temp = str(path) + '.tmp'
    torch.save(obj, temp); os.replace(temp, path)


os.makedirs(OUT, exist_ok=True)
folds = assign_folds(df, LABEL_COLS, N_FOLDS, SEED, GROUP_COL)
fold_table = df[['StudyInstanceUID', 'source']].assign(fold=folds)
fold_table.to_csv(os.path.join(OUT, 'folds.csv'), index=False)
# Record the dataset identity so resumed folds cannot silently use changed labels or cache.
import hashlib, importlib.metadata
identity = hashlib.sha256((df.to_json(orient='split') + json.dumps(CACHE_META, sort_keys=True)
                           + fold_table.to_json()).encode()).hexdigest()
recipe = dict(backbone=BACKBONE, img=IMG, n_slices=N_SLICES, planes=PLANES,
              label_cols=LABEL_COLS, t_train=T_TRAIN, preprocessing=CACHE_META,
              normalization=NORMALIZATION, drop=DROPOUT, seed=SEED,
              epochs=EPOCHS, lr=LR, weight_decay=WD, n_folds=N_FOLDS, accum_steps=ACCUM_STEPS,
              encoder_lr_scale=ENCODER_LR_SCALE, batch=BATCH, identity=identity,
              versions={p: importlib.metadata.version(p) for p in ('torch','timm','numpy')})
with open(os.path.join(OUT, 'run_config.json'), 'w') as handle: json.dump(recipe, handle, indent=2)
all_hist = []
for fold in FOLDS_TO_RUN:
    assert fold in range(N_FOLDS)
    fold_seed = SEED + fold
    torch.manual_seed(fold_seed); np.random.seed(fold_seed); random.seed(fold_seed)
    tr_rows = np.flatnonzero((folds >= 0) & (folds != fold))
    va_rows = np.flatnonzero(folds == fold)
    assert not set(tr_rows) & set(va_rows)
    assert df.iloc[tr_rows]['source'].eq('pseudo').all()
    dl_tr = make_loader(tr_rows, True, fold_seed)
    dl_val = make_loader(va_rows, False, fold_seed)
    model = KneeNet(BACKBONE, len(LABEL_COLS), len(PLANES), pretrained=not (RESUME and os.path.exists(os.path.join(OUT, f'last_fold_{fold}.pt'))),
                    drop=DROPOUT, img=IMG, normalization=NORMALIZATION).to(DEV)
    head_params = [p for name,p in model.named_parameters() if not name.startswith('enc.')]
    opt = torch.optim.AdamW([{'params':model.enc.parameters(),'lr':LR*ENCODER_LR_SCALE},
                            {'params':head_params,'lr':LR}], weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR*ENCODER_LR_SCALE, LR],
                    total_steps=EPOCHS*math.ceil(len(dl_tr)/ACCUM_STEPS), pct_start=.1)
    scaler = torch.amp.GradScaler(DEV, enabled=DEV=='cuda')
    path = os.path.join(OUT, f'fold_{fold}.pt')
    last_path = os.path.join(OUT, f'last_fold_{fold}.pt')
    start, best, hist = 0, float('inf'), []
    if RESUME and os.path.exists(last_path):
        saved = torch.load(last_path, map_location='cpu', weights_only=False)
        assert saved['cfg'] == recipe, 'Resume recipe differs; use a new OUT directory'
        model.load_state_dict(saved['model']); opt.load_state_dict(saved['optimizer'])
        sched.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler'])
        start, best, hist = saved['epoch']+1, saved['best'], saved['history']
        assert os.path.exists(path), 'Resume requires both last and best fold checkpoints'
    elif os.path.exists(path):
        raise RuntimeError(f'{path} already exists; set RESUME=True or choose an empty OUT')
    for ep in range(start, EPOCHS):
        # Epoch-local worker seeds make interrupted runs repeatable at epoch boundaries.
        epoch_seed = fold_seed*1000 + ep
        torch.manual_seed(epoch_seed); np.random.seed(epoch_seed); random.seed(epoch_seed)
        dl_tr = make_loader(tr_rows, True, epoch_seed)
        model.train(); total, count = 0., 0; t0 = time.time()
        opt.zero_grad(set_to_none=True)
        for step, (x, present, y) in enumerate(tqdm(dl_tr, desc=f'fold {fold} epoch {ep+1}')):
            x, present, y = x.to(DEV, non_blocking=True), present.to(DEV), y.to(DEV)
            logits, _ = model(x, present)
            loss = masked_bce(logits.float(), y)
            if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss')
            group_start = (step // ACCUM_STEPS) * ACCUM_STEPS
            group_size = min(ACCUM_STEPS, len(dl_tr)-group_start)
            scaler.scale(loss/group_size).backward()
            if (step+1) % ACCUM_STEPS == 0 or step+1 == len(dl_tr):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
                old_scale = scaler.get_scale()
                scaler.step(opt); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
                opt.zero_grad(set_to_none=True)
            total += loss.item()*len(x); count += len(x)
        yv, pv = evaluate(model, dl_val)
        observed = np.isfinite(yv)
        clipped = np.clip(pv, 1e-7, 1-1e-7)
        val_loss = float((-(np.nan_to_num(yv)*np.log(clipped) +
                           (1-np.nan_to_num(yv))*np.log1p(-clipped)))[observed].mean())
        auc, _ = macro_auc(yv, pv, LABEL_COLS)
        rec = dict(fold=fold, epoch=ep, loss=total/count, val_bce=val_loss,
                   auc_pseudo=auc, minutes=(time.time()-t0)/60)
        hist.append(rec); print(rec)
        # Soft-label BCE selects epochs; the human gold set is never consulted.
        if val_loss < best:
            best = val_loss
            atomic_torch_save(dict(model=model.state_dict(), cfg=recipe, epoch=ep,
                                   fold=fold, val_bce=best, auc_pseudo=auc), path)
        atomic_torch_save(dict(model=model.state_dict(), cfg=recipe, epoch=ep, best=best,
                               optimizer=opt.state_dict(), scheduler=sched.state_dict(),
                               scaler=scaler.state_dict(), history=hist), last_path)
        pd.DataFrame(hist).to_csv(os.path.join(OUT, f'history_fold_{fold}.csv'), index=False)
    model.load_state_dict(torch.load(path, map_location=DEV, weights_only=False)['model'])
    yv, pv = evaluate(model, dl_val)
    output = pd.DataFrame(pv, columns=LABEL_COLS)
    output.insert(0,'StudyInstanceUID',df.iloc[va_rows]['StudyInstanceUID'].to_numpy())
    output['fold'] = fold
    output.to_csv(os.path.join(OUT,f'oof_fold_{fold}.csv'), index=False)
    gold_rows = np.flatnonzero(folds == -1)
    yg, pg = evaluate(model, make_loader(gold_rows, False, fold_seed))
    gold_output = pd.DataFrame(pg, columns=LABEL_COLS)
    gold_output.insert(0,'StudyInstanceUID',df.iloc[gold_rows]['StudyInstanceUID'].to_numpy())
    gold_output.to_csv(os.path.join(OUT,f'gold_fold_{fold}.csv'), index=False)
    print('Gold audit after checkpoint selection:',macro_auc(yg,pg,LABEL_COLS)[0])
    all_hist.extend(hist)
    del model, opt, sched, scaler, dl_tr, dl_val
    gc.collect()
    if DEV=='cuda': torch.cuda.empty_cache()

paths = [os.path.join(OUT,f'oof_fold_{f}.csv') for f in range(N_FOLDS)]
if all(os.path.exists(p) for p in paths):
    oof = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    expected = df.loc[folds>=0,'StudyInstanceUID']
    assert oof['StudyInstanceUID'].is_unique and set(oof['StudyInstanceUID']) == set(expected)
    aligned = oof.set_index('StudyInstanceUID').loc[expected]
    oof.to_csv(os.path.join(OUT,'oof_predictions.csv'), index=False)
    print('OOF pseudo AUC (noisy-label proxy, selection-biased):',
          macro_auc(df.loc[folds>=0,LABEL_COLS].to_numpy(),aligned[LABEL_COLS].to_numpy(),LABEL_COLS))
else:
    print('Partial fold run; complete all folds before comparing aggregate OOF results.')

# Fold predictions share the same gold study order; audit the exact inference blend.
gold_paths = [os.path.join(OUT,f'gold_fold_{f}.csv') for f in range(N_FOLDS)]
if all(os.path.exists(p) for p in gold_paths):
    gold_ids = df.loc[folds==-1,'StudyInstanceUID']
    if len(gold_ids):
        members = [pd.read_csv(p).set_index('StudyInstanceUID').loc[gold_ids,LABEL_COLS].to_numpy()
                   for p in gold_paths]
        blended = rank_blend(members)
        audit = pd.DataFrame(blended,columns=LABEL_COLS)
        audit.insert(0,'StudyInstanceUID',gold_ids.to_numpy())
        audit.to_csv(os.path.join(OUT,'gold_ensemble.csv'),index=False)
        print('Gold fold-ensemble audit:', macro_auc(df.loc[folds==-1,LABEL_COLS].to_numpy(),blended,LABEL_COLS))

## 5 · Per-finding results

In [ ]:
# Run cells in order; no working-directory changes are needed.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

## 6 · Optional: throughput check

Run this if training feels slow. It separates data from compute, which is the only
way to tell them apart — guessing cost a wasted run once already.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

## Outputs

Attach fold_0.pt through fold_4.pt to submission. Keep last_fold_N.pt for resuming training; inference ignores those files. Kaggle reruns start with empty working storage: copy prior outputs to OUT before setting RESUME=True. Do not mix recipes in one checkpoint directory.